In [2]:
# Paste this WHOLE artifact into ONE ordinary notebook cell and run it.
# DO NOT put %%writefile above it. This cell writes and launches the app itself.

from pathlib import Path
import ast
import importlib.util
import os
import socket
import subprocess
import sys
import time
import urllib.request
import webbrowser
from datetime import datetime

APP_SOURCE = r'''
"""Local PM2.5 hotspot screening and interactive regional exploration."""
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import timedelta
import contextlib
import html
import io
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.neighbors import BallTree
from scipy.spatial import cKDTree, Delaunay, QhullError
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import folium
from folium.raster_layers import ImageOverlay
from folium.utilities import mercator_transform
from streamlit_folium import st_folium
import streamlit as st

REPO_ROOT = Path(__file__).resolve().parents[1]  # the app file is written to <repo>/output/
EARTH_KM = 6371.0088
SECTORS = np.array(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])
COLORS = ["#37688c", "#68a4ad", "#a6d3a4", "#add47b", "#e1dc84", "#dba85e", "#a86646"]
SURFACE_CMAP = LinearSegmentedColormap.from_list("regional_pm", COLORS)

@dataclass
class Config:
    sensor_path: str = str(REPO_ROOT / "data" / "data_sensors.parquet")
    weather_path: str = str(REPO_ROOT / "data" / "knmi_weather_data.parquet")
    output_dir: str = str(REPO_ROOT / "output" / "pm25_hotspot_results")
    batch_size: int = 100_000
    start: str = ""
    end: str = ""
    initial_days: int = 14
    sensor_tz: str = "UTC"
    weather_tz: str = "UTC"
    sensor_epoch_unit: str = ""
    weather_epoch_unit: str = ""
    weather_interval_end: bool = True
    radius_km: float = 5.0
    max_neighbors: int = 30
    min_neighbors: int = 3
    min_readings: int = 1
    min_excess: float = 5.0
    robust_z: float = 2.5
    scale_floor: float = 2.0
    max_weather_km: float = 25.0
    resultant_min: float = 0.5
    upwind_angle: float = 45.0
    min_upwind: int = 2
    temporal_hours: int = 3
    sustained_hours: int = 3
    persistent_fraction: float = 0.20
    persistence_hours: int = 24
    persistence_coverage: float = 0.50
    persistence_days: int = 3
    save_outputs: bool = True


def parse_time(series, naive_tz="UTC", epoch_unit=""):
    """Aware times retain their timezone; naive times use the chosen zone."""
    if epoch_unit:
        return pd.to_datetime(pd.to_numeric(series, errors="coerce"), unit=epoch_unit, errors="coerce", utc=True)
    text = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns, UTC]")
    aware = text.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", case=False, na=False)
    out.loc[aware] = pd.to_datetime(text.loc[aware], format="mixed", errors="coerce", utc=True)
    naive = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    compact = pd.Series(False, index=series.index)
    for length, fmt in {8: "%Y%m%d", 10: "%Y%m%d%H", 12: "%Y%m%d%H%M", 14: "%Y%m%d%H%M%S"}.items():
        mask = (~aware) & text.str.fullmatch(rf"\d{{{length}}}", na=False)
        compact |= mask
        if length == 10:
            day = pd.to_datetime(text.loc[mask].str[:8], format="%Y%m%d", errors="coerce")
            hour = pd.to_numeric(text.loc[mask].str[8:], errors="coerce")
            naive.loc[mask] = day + pd.to_timedelta(hour.where(hour.between(0, 24)), unit="h")
        else:
            naive.loc[mask] = pd.to_datetime(text.loc[mask], format=fmt, errors="coerce")
    other = ~aware & ~compact
    naive.loc[other] = pd.to_datetime(text.loc[other], format="mixed", errors="coerce")
    out.loc[~aware] = naive.loc[~aware].dt.tz_localize(naive_tz, ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")
    return out


def parquet(path, columns):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File not found on the computer running Streamlit: {p}")
    pf = pq.ParquetFile(p)
    missing = set(columns) - set(pf.schema_arrow.names)
    if missing:
        raise ValueError(f"{p.name}: missing columns {sorted(missing)}")
    return pf


def time_range(path, col, tz, unit, shift, batch_size):
    pf = parquet(path, [col])
    field = pf.schema_arrow.field(col)
    typed = pa.types.is_timestamp(field.type) or pa.types.is_date(field.type)
    index = pf.schema_arrow.get_field_index(col)
    bounds = []
    if typed:
        for rg in range(pf.num_row_groups):
            stat = pf.metadata.row_group(rg).column(index).statistics
            if stat is None or not stat.has_min_max:
                bounds = []
                break
            bounds.extend([stat.min, stat.max])
    if bounds:
        times = parse_time(pd.Series(bounds), tz, unit).dropna() + shift
        if not times.empty:
            print(f"{Path(path).name}: {pf.metadata.num_rows:,} rows; metadata timestamp range")
            return times.min(), times.max()
    lo, hi, bad = None, None, 0
    print(f"Scanning timestamps in {Path(path).name}...")
    for ab in pf.iter_batches(batch_size=batch_size, columns=[col]):
        raw = ab.to_pandas()[col]
        times = parse_time(raw, tz, unit) + shift
        bad += int((raw.notna() & times.isna()).sum())
        times = times.dropna()
        if not times.empty:
            lo = times.min() if lo is None else min(lo, times.min())
            hi = times.max() if hi is None else max(hi, times.max())
    print(f"Unparseable nonmissing timestamps: {bad:,}")
    if lo is None:
        raise ValueError(f"No usable timestamps in {path}. Check timezone and epoch-unit settings.")
    return lo, hi


def batches(path, columns, time_col, tz, unit, shift, start, end, batch_size):
    pf = parquet(path, columns)
    field = pf.schema_arrow.field(time_col)
    typed = pa.types.is_timestamp(field.type) or pa.types.is_date(field.type)
    index = pf.schema_arrow.get_field_index(time_col)
    read, retained = 0, 0
    for rg in range(pf.num_row_groups):
        stat = pf.metadata.row_group(rg).column(index).statistics if typed else None
        if stat is not None and stat.has_min_max:
            bound = parse_time(pd.Series([stat.min, stat.max]), tz, unit) + shift
            if bound.notna().all() and (bound.iloc[1] < start or bound.iloc[0] >= end):
                continue
        for ab in pf.iter_batches(batch_size=batch_size, columns=columns, row_groups=[rg]):
            df = ab.to_pandas()
            times = parse_time(df[time_col], tz, unit) + shift
            keep = times.ge(start) & times.lt(end)
            read += len(df)
            if keep.any():
                selected = df.loc[keep].copy()
                selected["hour"] = times.loc[keep].dt.floor("h")
                retained += len(selected)
                yield selected
    print(f"{Path(path).name}: read {read:,} rows; retained {retained:,} in window")


def add(total, current):
    return current if total is None else total.add(current, fill_value=0)


def aggregate_sensors(c, start, end):
    sums, coords = None, None
    qc = {"missing_sensor_id": 0, "invalid_coordinates": 0, "invalid_or_negative_pm": 0}
    cols = ["kit_id", "timestamp_from", "pm25_raw", "latitude", "longitude"]
    for b in batches(c.sensor_path, cols, "timestamp_from", c.sensor_tz, c.sensor_epoch_unit, pd.Timedelta(0), start, end, c.batch_size):
        b["sensor"] = b.kit_id.astype("string").str.strip()
        has_id = b.sensor.notna() & b.sensor.ne("")
        qc["missing_sensor_id"] += int((~has_id).sum())
        b = b.loc[has_id].copy()
        for col in ["latitude", "longitude", "pm25_raw"]:
            b[col] = pd.to_numeric(b[col], errors="coerce")
        geo = b.latitude.between(-90, 90) & b.longitude.between(-180, 180)
        qc["invalid_coordinates"] += int((~geo).sum())
        g = b.loc[geo, ["sensor", "latitude", "longitude"]].copy()
        g["n"] = 1
        coords = add(coords, g.groupby("sensor")[["latitude", "longitude", "n"]].sum())
        good_pm = np.isfinite(b.pm25_raw) & b.pm25_raw.ge(0)
        qc["invalid_or_negative_pm"] += int((~good_pm).sum())
        g = b.loc[good_pm, ["sensor", "hour", "pm25_raw"]].copy()
        g["reading_count"] = 1
        sums = add(sums, g.groupby(["sensor", "hour"])[["pm25_raw", "reading_count"]].sum())
    print("Sensor QC:", qc)
    if sums is None or sums.empty or coords is None or coords.empty:
        raise ValueError("No usable sensor readings and coordinates in this window.")
    loc = coords.assign(lat=coords.latitude / coords.n, lon=coords.longitude / coords.n)[["lat", "lon"]]
    h = sums.reset_index()
    h["pm"] = h.pm25_raw / h.reading_count
    h = h.loc[(h.reading_count >= c.min_readings) & h.sensor.isin(loc.index)]
    if h.empty:
        raise ValueError("No sensor-hours meet the reading-count and location requirements.")
    loc = loc.loc[loc.index.isin(h.sensor)]
    return h[["sensor", "hour", "pm", "reading_count"]], loc


def aggregate_weather(c, start, end, shift):
    empty = pd.DataFrame(columns=["station", "hour", "dd", "wind_status"])
    empty_loc = pd.DataFrame(columns=["lat", "lon"])
    if not c.weather_path.strip():
        print("Weather omitted: concentration screening will continue without wind features.")
        return empty, empty_loc
    total, coords = None, None
    cols = ["station", "time", "lat", "lon", "DD"]
    for b in batches(c.weather_path, cols, "time", c.weather_tz, c.weather_epoch_unit, shift, start, end, c.batch_size):
        b["station"] = b.station.astype("string").str.strip()
        b = b.loc[b.station.notna() & b.station.ne("")].copy()
        for col in ["lat", "lon", "DD"]:
            b[col] = pd.to_numeric(b[col], errors="coerce")
        geo = b.lat.between(-90, 90) & b.lon.between(-180, 180)
        g = b.loc[geo, ["station", "lat", "lon"]].copy()
        g["n"] = 1
        coords = add(coords, g.groupby("station")[["lat", "lon", "n"]].sum())
        valid = b.DD.between(1, 360)
        rad = np.deg2rad(b.DD.where(valid, 0))
        g = b[["station", "hour"]].copy()
        g["sin"] = np.where(valid, np.sin(rad), 0)
        g["cos"] = np.where(valid, np.cos(rad), 0)
        g["directional"] = valid.astype(int)
        g["calm"] = b.DD.eq(0).astype(int)
        g["variable"] = b.DD.eq(990).astype(int)
        total = add(total, g.groupby(["station", "hour"]).sum(numeric_only=True))
    if total is None or total.empty or coords is None or coords.empty:
        print("No located weather observations in this window; wind features will be unavailable.")
        return empty, empty_loc
    loc = coords[["lat", "lon"]].div(coords.n, axis=0)
    w = total.reset_index()
    resultant = np.hypot(w["sin"], w["cos"]) / w.directional.replace(0, np.nan)
    directional = (w.directional > 0) & (w.calm == 0) & (w.variable == 0) & (resultant >= c.resultant_min)
    w["dd"] = ((np.rad2deg(np.arctan2(w["sin"], w["cos"])) + 360) % 360).where(directional)
    w["wind_status"] = "missing_or_nonstandard"
    w.loc[(w.calm > 0) & (w.directional == 0) & (w.variable == 0), "wind_status"] = "calm"
    w.loc[(w.variable > 0) | ((w.directional > 0) & ~directional), "wind_status"] = "variable_or_mixed"
    w.loc[directional, "wind_status"] = "directional"
    print(f"Weather: {len(w):,} station-hours")
    return w[["station", "hour", "dd", "wind_status"]], loc


def neighbor_graph(loc, c):
    n = len(loc)
    coordinates = loc[["lat", "lon"]].to_numpy()
    tree = BallTree(np.deg2rad(coordinates), metric="haversine")
    dist, ids = tree.query(np.deg2rad(coordinates), k=min(c.max_neighbors + 1, n))
    graph = []
    for i in range(n):
        keep = (ids[i] != i) & (dist[i] * EARTH_KM <= c.radius_km)
        j = ids[i][keep][:c.max_neighbors]
        d = (dist[i][keep] * EARTH_KM)[:c.max_neighbors]
        dx = EARTH_KM * np.cos(np.deg2rad(coordinates[i, 0])) * np.deg2rad(coordinates[j, 1] - coordinates[i, 1])
        dy = EARTH_KM * np.deg2rad(coordinates[j, 0] - coordinates[i, 0])
        graph.append((j, d, dx, dy))
    return graph


def local_plane(dx, dy, distances, values):
    """Weighted neighbor-only plane: intercept, east slope, north slope."""
    if len(values) < 4:
        return np.full(3, np.nan)
    scale = max(float(np.median(distances)), 0.1)
    X = np.column_stack([np.ones(len(values)), dx / scale, dy / scale])
    root_w = 1 / (distances + 0.25)
    A = X * root_w[:, None]
    if np.linalg.matrix_rank(A) < 3 or np.linalg.cond(A) > 100:
        return np.full(3, np.nan)
    coef = np.linalg.lstsq(A, values * root_w, rcond=None)[0]
    return np.array([coef[0], coef[1] / scale, coef[2] / scale])


def score_hours(c, hourly, locations, weather, weather_locations, start, end, progress=None):
    loc = locations.sort_index().copy()
    sensors = loc.index.to_numpy()
    indices = {name: i for i, name in enumerate(sensors)}
    graph = neighbor_graph(loc, c)
    n = len(sensors)
    loc["weather_station"] = pd.NA
    loc["weather_distance_km"] = np.nan
    if not weather_locations.empty:
        tree = BallTree(np.deg2rad(weather_locations[["lat", "lon"]].to_numpy()), metric="haversine")
        distance, idx = tree.query(np.deg2rad(loc[["lat", "lon"]].to_numpy()), k=1)
        loc["weather_station"] = weather_locations.index.to_numpy()[idx[:, 0]]
        loc["weather_distance_km"] = distance[:, 0] * EARTH_KM
    station_names = loc.weather_station.to_numpy()
    station_distances = loc.weather_distance_km.to_numpy(dtype=float)
    lookup = {(row.station, row.hour): (row.dd, row.wind_status) for row in weather.itertuples(index=False)}
    hourly = hourly.assign(i=hourly.sensor.map(indices))
    groups = {t: g for t, g in hourly.groupby("hour")}
    times = pd.date_range(start, end, freq="h", inclusive="left")
    pm_hist = np.full((c.temporal_hours, n), np.nan)
    excess_hist = np.full_like(pm_hist, np.nan)
    tx = np.arange(c.temporal_hours, dtype=float)
    tx -= tx.mean()
    denom = tx @ tx
    records = []
    for count, hour in enumerate(times, 1):
        pm = np.full(n, np.nan)
        reading_counts = np.zeros(n, dtype=int)
        g = groups.get(hour)
        if g is not None:
            ii = g.i.to_numpy(dtype=int)
            pm[ii] = g.pm.to_numpy()
            reading_counts[ii] = g.reading_count.to_numpy(dtype=int)
        pm_hist = np.roll(pm_hist, -1, axis=0)
        pm_hist[-1] = pm
        excess = np.full(n, np.nan)
        hour_rows = []
        for i in np.flatnonzero(np.isfinite(pm)):
            j, d, dx, dy = graph[i]
            present = np.isfinite(pm[j])
            j, d, dx, dy = j[present], d[present], dx[present], dy[present]
            values = pm[j]
            eligible = len(values) >= c.min_neighbors
            background = float(np.median(values)) if eligible else np.nan
            mad = float(np.median(np.abs(values - background))) if eligible else np.nan
            delta = pm[i] - background
            z = delta / max(1.4826 * mad, c.scale_floor) if eligible else np.nan
            excess[i] = delta
            plane = local_plane(dx, dy, d, values) if eligible else np.full(3, np.nan)
            station, weather_km = station_names[i], station_distances[i]
            if pd.notna(station):
                dd, status = lookup.get((station, hour), (np.nan, "missing_weather_hour"))
            else:
                dd, status = np.nan, "no_weather_station"
            if np.isfinite(weather_km) and weather_km > c.max_weather_km:
                dd, status = np.nan, "weather_station_too_far"
            sector, support, along_slope = status, pd.NA, np.nan
            upstream_excess, n_upwind = np.nan, 0
            if status == "directional":
                theta = np.deg2rad(dd)
                east, north = -np.sin(theta), -np.cos(theta)
                along_slope = plane[1] * east + plane[2] * north
                along = dx * east + dy * north
                cross = np.abs(-dx * north + dy * east)
                upwind = (along < -0.1) & (cross <= -along * np.tan(np.deg2rad(c.upwind_angle)))
                n_upwind = int(upwind.sum())
                sector = SECTORS[int(((dd + 22.5) % 360) // 45)]
                if n_upwind >= c.min_upwind:
                    upstream_excess = pm[i] - float(np.median(values[upwind]))
                    support = bool(upstream_excess >= c.min_excess)
            history = pm_hist[:, i]
            temporal = float(history @ tx / denom) if np.isfinite(history).all() else np.nan
            hour_rows.append(dict(sensor=sensors[i], hour=hour, pm=pm[i], reading_count=reading_counts[i],
                neighbor_count=len(values), eligible=eligible, local_background=background, local_excess=delta,
                robust_z=z, hotspot=bool(eligible and delta >= c.min_excess and z >= c.robust_z),
                background_plane_at_sensor=plane[0], spatial_slope_east=plane[1], spatial_slope_north=plane[2],
                temporal_slope=temporal, weather_station=station, weather_distance_km=weather_km, dd=dd,
                wind_status=status, wind_sector=sector, downwind_background_slope=along_slope,
                upwind_neighbor_count=n_upwind, upwind_excess=upstream_excess, wind_support=support))
        excess_hist = np.roll(excess_hist, -1, axis=0)
        excess_hist[-1] = excess
        for row in hour_rows:
            history = excess_hist[:, indices[row["sensor"]]]
            row["temporal_excess_slope"] = float(history @ tx / denom) if np.isfinite(history).all() else np.nan
        records.extend(hour_rows)
        if count % 24 == 0 or count == len(times):
            print(f"Scored {count:,}/{len(times):,} hours")
            if progress is not None:
                progress(count / len(times))
    if not records:
        raise ValueError("No sensor-hours were available for scoring.")
    return pd.DataFrame(records), loc, len(times)


def classify(c, result, loc, total_hours):
    result = result.sort_values(["sensor", "hour"]).reset_index(drop=True).copy()
    result["event_id"] = pd.NA
    result["event_class"] = "not_flagged"
    events = []
    for sensor, group in result.groupby("sensor", sort=False):
        new_run = group.hour.diff().ne(pd.Timedelta(hours=1)) | group.hotspot.ne(group.hotspot.shift())
        for _, run in group.groupby(new_run.cumsum()):
            if not run.hotspot.iloc[0]:
                continue
            label = "sustained" if len(run) >= c.sustained_hours else "short_lived"
            eid = f"event_{len(events)+1:06d}"
            result.loc[run.index, "event_id"] = eid
            result.loc[run.index, "event_class"] = label
            events.append(dict(event_id=eid, sensor=sensor, start=run.hour.min(),
                end_exclusive=run.hour.max()+pd.Timedelta(hours=1), hours=len(run), event_class=label,
                mean_excess=run.local_excess.mean(), peak_excess=run.local_excess.max(),
                wind_comparable_hours=int(run.wind_support.notna().sum()),
                wind_supported_hours=int(run.wind_support.eq(True).sum())))
    summary = result.groupby("sensor").agg(observed_hours=("hour", "size"), eligible_hours=("eligible", "sum"),
        hotspot_hours=("hotspot", "sum"), mean_pm=("pm", "mean"), max_excess=("local_excess", "max"))
    days = result.loc[result.hotspot].assign(day=lambda x: x.hour.dt.floor("D")).groupby("sensor").day.nunique()
    summary["hotspot_days"] = days.reindex(summary.index, fill_value=0)
    summary["hotspot_fraction"] = summary.hotspot_hours / summary.eligible_hours.replace(0, np.nan)
    summary["assessment_coverage"] = summary.eligible_hours / total_hours
    assessed = (summary.eligible_hours >= c.persistence_hours) & (summary.assessment_coverage >= c.persistence_coverage)
    recurring = assessed & (summary.hotspot_fraction >= c.persistent_fraction) & (summary.hotspot_days >= c.persistence_days)
    summary["persistence_class"] = np.where(~assessed, "insufficient_coverage", np.where(recurring, "recurring_hotspot", "not_recurring_by_rule"))
    summary = summary.join(loc).sort_values(["hotspot_hours", "eligible_hours"], ascending=False)
    cols = ["event_id", "sensor", "start", "end_exclusive", "hours", "event_class", "mean_excess", "peak_excess", "wind_comparable_hours", "wind_supported_hours"]
    events = pd.DataFrame(events, columns=cols)
    wind = result.loc[result.eligible & result.wind_status.eq("directional")].groupby("wind_sector").agg(
        assessed_sensor_hours=("hour", "size"), hotspot_hours=("hotspot", "sum"), median_excess=("local_excess", "median"))
    wind["hotspot_fraction"] = wind.hotspot_hours / wind.assessed_sensor_hours
    return result, summary, events, wind.reindex(SECTORS)


def plot_sensor(result, sensor):
    g = result.loc[result.sensor.eq(sensor)].set_index("hour").sort_index()
    full = g.reindex(pd.date_range(g.index.min(), g.index.max(), freq="h"))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(full.index, full.pm, label="Sensor hourly mean")
    ax.plot(full.index, full.local_background, label="Nearby-sensor median", alpha=0.8)
    for name, color in [("short_lived", "orange"), ("sustained", "red")]:
        hit = g.loc[g.event_class.eq(name)]
        ax.scatter(hit.index, hit.pm, color=color, s=25, label=name.replace("_", " "))
    ax.set(title=f"Sensor {sensor}", xlabel="Time (UTC)", ylabel="PM2.5 (reported units)")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    return fig


def run_pipeline(c, progress=None):
    if c.max_neighbors < c.min_neighbors or c.temporal_hours < 2 or c.scale_floor <= 0 or c.radius_km <= 0:
        raise ValueError("Check neighbour, temporal-window, radius and scale-floor settings.")
    shift = pd.Timedelta(hours=-1 if c.weather_interval_end else 0)
    s0, s1 = time_range(c.sensor_path, "timestamp_from", c.sensor_tz, c.sensor_epoch_unit, pd.Timedelta(0), c.batch_size)
    first = s0.floor("h")
    last = s1.floor("h") + pd.Timedelta(hours=1)
    # The time window is sensor-based: missing weather must not remove PM hours.
    start = parse_time(pd.Series([c.start]), c.sensor_tz).iloc[0].floor("h") if c.start.strip() else first
    end = parse_time(pd.Series([c.end]), c.sensor_tz).iloc[0].floor("h") if c.end.strip() else min(start + pd.Timedelta(days=c.initial_days), last)
    if pd.isna(start) or pd.isna(end) or end <= start or start >= last or end <= first:
        raise ValueError("No usable sensor window. Check start/end, timezone and timestamp format.")
    start, end = max(start, first), min(end, last)
    print(f"Analysis window [start, end): {start} to {end}")
    print(f"Weather timestamp shift: {shift}; confirm this convention against the export.")
    hourly, locations = aggregate_sensors(c, start, end)
    weather, weather_loc = aggregate_weather(c, start, end, shift)
    result, loc, hours = score_hours(c, hourly, locations, weather, weather_loc, start, end, progress)
    result, summary, events, wind = classify(c, result, loc, hours)
    if c.save_outputs:
        out = Path(c.output_dir)
        out.mkdir(parents=True, exist_ok=True)
        result.to_csv(out / "sensor_hour_features.csv", index=False)
        summary.to_csv(out / "sensor_hotspot_summary.csv", index_label="sensor")
        events.to_csv(out / "hotspot_events.csv", index=False)
        wind.to_csv(out / "wind_sector_summary.csv", index_label="wind_from_sector")
        settings = [*(f"{k} = {v}" for k, v in asdict(c).items()), f"actual_start = {start}", f"actual_end = {end}"]
        (out / "run_settings.txt").write_text("\n".join(settings), encoding="utf-8")
        fig = plot_sensor(result, summary.index[0])
        fig.savefig(out / "example_sensor_timeseries.png", dpi=180)
        plt.close(fig)
        print(f"Saved CSV outputs to {out}")
    print(f"Sensor-hours: {len(result):,}; assessable: {int(result.eligible.sum()):,}; flagged: {int(result.hotspot.sum()):,}")
    print("Wind statuses:\n", result.wind_status.value_counts().to_string())
    return dict(features=result, summary=summary, events=events, wind=wind,
                start=start, end=end, config=asdict(c))


def region_xy(lat, lon, center_lat, center_lon):
    """Local east/north distances in km; small-region approximation."""
    lat, lon = np.asarray(lat, dtype=float), np.asarray(lon, dtype=float)
    return (EARTH_KM * np.cos(np.deg2rad(center_lat)) * np.deg2rad(lon-center_lon),
            EARTH_KM * np.deg2rad(lat-center_lat))


def make_surface(hour_data, center_lat, center_lon, radius, support, field, lower, upper, pixels=180):
    """IDW interpolation with a three-distinct-site and convex-hull mask."""
    d = hour_data[["lat", "lon", field]].copy()
    for col in ["lat", "lon", field]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if d.empty:
        return None
    x, y = region_xy(d.lat, d.lon, center_lat, center_lon)
    d = d.loc[np.hypot(x, y) <= radius + support]
    d = d.groupby(["lat", "lon"], as_index=False)[field].mean()
    if len(d) < 3:
        return None
    x, y = region_xy(d.lat, d.lon, center_lat, center_lon)
    sites = np.column_stack([x, y])
    try:
        hull = Delaunay(sites)
    except QhullError:
        return None
    lat_pad = np.rad2deg(radius/EARTH_KM)
    lon_pad = np.rad2deg(radius/(EARTH_KM*np.cos(np.deg2rad(center_lat))))
    south, north = center_lat-lat_pad, center_lat+lat_pad
    west, east = center_lon-lon_pad, center_lon+lon_pad
    lng, lat = np.meshgrid(np.linspace(west, east, pixels), np.linspace(north, south, pixels))
    gx, gy = region_xy(lat, lng, center_lat, center_lon)
    points = np.column_stack([gx.ravel(), gy.ravel()])
    inside = (np.hypot(points[:, 0], points[:, 1]) <= radius) & (hull.find_simplex(points) >= 0)
    distances, indices = cKDTree(sites).query(points, k=min(12, len(sites)), distance_upper_bound=support)
    present = np.isfinite(distances) & (indices < len(sites))
    weights = np.where(present, 1/(distances**2 + 0.05**2), 0)
    values = d[field].to_numpy(dtype=float)[np.minimum(indices, len(sites)-1)]
    numerator, denominator = (weights*values).sum(axis=1), weights.sum(axis=1)
    z = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)
    visible = inside & (present.sum(axis=1) >= 3) & (denominator > 0)
    if not visible.any():
        return None
    rgba = SURFACE_CMAP(np.clip((z-lower)/(upper-lower), 0, 1)).reshape(pixels, pixels, 4)
    rgba[:, :, 3] = visible.reshape(pixels, pixels) * 0.78
    # Preserve exact colour values; Folium otherwise normalises float channels.
    image = mercator_transform(rgba, (south, north), origin="upper")
    image = (np.clip(image, 0, 1)*255).astype(np.uint8)
    return image, [[south, west], [north, east]]


def color_limits(data, field):
    values = pd.to_numeric(data[field], errors="coerce").to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (-1.0, 1.0) if field == "local_excess" else (0.0, 1.0)
    if field == "local_excess":
        bound = float(np.quantile(np.abs(values), 0.98))
        return (-bound, bound) if bound > 0 else (-1.0, 1.0)
    bound = float(np.quantile(np.maximum(values, 0), 0.98))
    return (0.0, bound if bound > 0 else 1.0)


def move_hour(delta, labels):
    current = st.session_state.get("pm25_hour", labels[0])
    i = labels.index(current) if current in labels else 0
    st.session_state["pm25_hour"] = labels[int(np.clip(i+delta, 0, len(labels)-1))]


def regional_overview(a):
    r, summary = a["features"], a["summary"]
    cols = st.columns(4)
    cols[0].metric("Sensors in run", f"{len(summary):,}")
    cols[1].metric("Assessable sensor-hours", f"{int(r.eligible.sum()):,}")
    cols[2].metric("Flagged sensor-hours", f"{int(r.hotspot.sum()):,}")
    cols[3].metric("Recurring candidates", f"{int(summary.persistence_class.eq('recurring_hotspot').sum()):,}")
    if not r.eligible.any():
        st.warning("No hours meet the neighbour requirements. Concentrations can still be viewed, but hotspot screening is not assessable.")
    elif r.eligible.mean() < 0.5:
        st.warning("Fewer than half of observed sensor-hours are assessable. Unassessable does not mean clean.")
    locations = summary.reset_index()[["sensor", "lat", "lon"]].dropna().copy()
    if locations.empty:
        st.info("No located sensors to display.")
        return
    if "pm25_center" not in st.session_state:
        site = locations.iloc[0]
        st.session_state["pm25_center"] = (float(site.lat), float(site.lon))
    center_lat, center_lon = st.session_state["pm25_center"]
    st.subheader("1. Select a local region")
    st.caption("Click the Netherlands map to set the centre. Alternatively, choose a sensor below. Map tiles require an internet connection.")
    national = folium.Map(location=[52.1, 5.3], zoom_start=7, tiles="OpenStreetMap", control_scale=True)
    shown = locations if len(locations) <= 2500 else locations.sample(2500, random_state=42)
    if len(shown) < len(locations):
        st.caption("Overview markers are sampled for responsiveness; local results use all available sensors.")
    for row in shown.itertuples(index=False):
        folium.CircleMarker([row.lat, row.lon], radius=2.5, color="#2b6b8a", fill=True,
            fill_opacity=0.7, tooltip=html.escape(str(row.sensor))).add_to(national)
    folium.CircleMarker([center_lat, center_lon], radius=8, color="#c03a2b", fill=False,
        tooltip="Current region centre").add_to(national)
    clicked = st_folium(national, height=360, use_container_width=True, key="pm25_national", returned_objects=["last_clicked"])
    mode = st.radio("Choose centre using", ["Map click", "Sensor location"], horizontal=True)
    if mode == "Sensor location":
        selected = st.selectbox("Centre on sensor", locations.sensor.tolist(), key="pm25_center_sensor")
        site = locations.loc[locations.sensor.eq(selected)].iloc[0]
        center_lat, center_lon = float(site.lat), float(site.lon)
    else:
        point = clicked.get("last_clicked") if clicked else None
        if point:
            center_lat, center_lon = float(point["lat"]), float(point["lng"])
    if not (45 <= center_lat <= 58 and -5 <= center_lon <= 15):
        st.warning("Choose a region near the Netherlands.")
        return
    st.session_state["pm25_center"] = (center_lat, center_lon)
    st.caption(f"Centre: {center_lat:.5f}, {center_lon:.5f}")
    left, right = st.columns(2)
    radius = left.slider("Displayed region radius (km)", 1.0, 60.0, 15.0, 1.0,
        help="Controls the map extent and regional table/list, not the screening neighbourhood.")
    support = right.slider("Interpolation support radius (km)", 0.5, 15.0, 3.0, 0.5,
        help="A pixel needs at least three distinct reporting locations within this distance and must lie inside the available sites' convex hull.")
    dx, dy = region_xy(locations.lat, locations.lon, center_lat, center_lon)
    region_ids = set(locations.loc[np.hypot(dx, dy) <= radius, "sensor"])
    buffer_ids = set(locations.loc[np.hypot(dx, dy) <= radius+support, "sensor"])
    st.subheader("2. Regional gradient map")
    choices = {
        "Local excess above nearby-sensor median": ("local_excess", "Local excess (reported PM units)"),
        "Hourly PM2.5 concentration": ("pm", "Hourly PM2.5 (reported units)"),
        "Spatial slope magnitude": ("spatial_slope_magnitude", "Spatial slope magnitude (reported PM units/km)"),
    }
    metric = st.selectbox("Map colour represents", list(choices))
    field, legend = choices[metric]
    hours = pd.date_range(a["start"], a["end"], freq="h", inclusive="left")
    labels = [t.strftime("%Y-%m-%d %H:%M UTC") for t in hours]
    if st.session_state.get("pm25_hour") not in labels:
        st.session_state["pm25_hour"] = labels[0]
    if len(labels) > 1:
        previous, slider, following = st.columns([1, 6, 1])
        previous.button("Previous", on_click=move_hour, args=(-1, labels), use_container_width=True)
        with slider:
            selected_label = st.select_slider("Time (UTC)", options=labels, key="pm25_hour")
        following.button("Next", on_click=move_hour, args=(1, labels), use_container_width=True)
    else:
        selected_label = labels[0]
    hour = hours[labels.index(selected_label)]
    st.caption(f"Hour beginning {selected_label}. Move the slider or use Previous / Next.")
    data = r.loc[r.sensor.isin(buffer_ids)].merge(locations, on="sensor", how="left")
    data["spatial_slope_magnitude"] = np.hypot(data.spatial_slope_east, data.spatial_slope_north)
    current = data.loc[data.hour.eq(hour)].copy()
    lower, upper = color_limits(data, field)
    surface = make_surface(current, center_lat, center_lon, radius, support, field, lower, upper)
    local = folium.Map(location=[center_lat, center_lon], tiles="OpenStreetMap", control_scale=True)
    lat_pad = np.rad2deg(radius/EARTH_KM)
    lon_pad = np.rad2deg(radius/(EARTH_KM*np.cos(np.deg2rad(center_lat))))
    local.fit_bounds([[center_lat-lat_pad, center_lon-lon_pad], [center_lat+lat_pad, center_lon+lon_pad]])
    if surface is not None:
        image, bounds = surface
        ImageOverlay(image=image, bounds=bounds, origin="upper", mercator_project=False,
            pixelated=True, name="Interpolated surface", opacity=0.9, interactive=False).add_to(local)
    folium.Circle([center_lat, center_lon], radius=radius*1000, color="#202020", weight=2,
                  fill=False, tooltip="Selected region").add_to(local)
    points = folium.FeatureGroup(name="Reporting sensors", show=True)
    for row in current.loc[current.sensor.isin(region_ids)].itertuples(index=False):
        excess_text = f"{row.local_excess:.2f}" if pd.notna(row.local_excess) else "not assessable"
        z_text = f"{row.robust_z:.2f}" if pd.notna(row.robust_z) else "not assessable"
        status = "Candidate hotspot" if row.hotspot else ("Assessable; not flagged" if row.eligible else "Not assessable")
        popup = (f"<b>Sensor:</b> {html.escape(str(row.sensor))}<br>"
            f"<b>Hour:</b> {selected_label}<br><b>PM2.5:</b> {row.pm:.2f}<br>"
            f"<b>Local excess:</b> {excess_text}<br><b>Robust score:</b> {z_text}<br>"
            f"<b>Status:</b> {status}<br><b>Coordinates:</b> {row.lat:.5f}, {row.lon:.5f}")
        folium.CircleMarker([row.lat, row.lon], radius=6 if row.hotspot else 4,
            color="#971b1b" if row.hotspot else "#193747", weight=2, fill=True,
            fill_color="#ee4b32" if row.hotspot else "#ffffff", fill_opacity=0.95,
            tooltip=html.escape(str(row.sensor)), popup=folium.Popup(popup, max_width=330)).add_to(points)
    points.add_to(local)
    folium.LayerControl(collapsed=False).add_to(local)
    # A new key per view prevents a stale overlay after time/region changes.
    detail_key = f"pm25_local_{center_lat:.5f}_{center_lon:.5f}_{radius}_{support}_{field}_{hour.isoformat()}"
    st_folium(local, height=540, use_container_width=True, key=detail_key, returned_objects=[])
    gradient = ", ".join(COLORS)
    st.markdown(
        f'<div style="max-width:650px;margin:5px 0 12px"><strong>{html.escape(legend)}</strong>'
        f'<div style="height:14px;background:linear-gradient(to right,{gradient});border-radius:3px"></div>'
        f'<div style="display:flex;justify-content:space-between;font-size:0.85em">'
        f'<span>{lower:.2f}</span><span>{(lower+upper)/2:.2f}</span><span>{upper:.2f}</span></div></div>',
        unsafe_allow_html=True)
    st.caption("The surface is interpolated, not measured at every pixel. Its scale is fixed across all hours in this run for the current region, buffer and metric. The 98th-percentile limit clips extremes. Blank areas have insufficient support; red sensor markers indicate flagged hours.")
    if surface is None:
        st.info("Not enough spatial support for this hour and metric. Try another hour, region, support radius or the concentration view.")
    regional_summary = summary.loc[summary.index.isin(region_ids)]
    st.subheader("Regional sensor summary")
    st.caption("This table covers the complete analysed time window, not just the selected hour.")
    st.dataframe(regional_summary.reset_index(), use_container_width=True)
    st.subheader("Possible hotspot candidates")
    hits = current.loc[current.sensor.isin(region_ids) & current.hotspot].sort_values("sensor")
    st.caption(f"Selected region; hour beginning {selected_label}. Coordinates are latitude, longitude. Candidates are not confirmed sources.")
    if hits.empty:
        st.write("No flagged candidates for this hour in the selected region. Missing or unassessable sensors are not evidence of clean air.")
    else:
        for row in hits.itertuples(index=False):
            st.text(f"Sensor {row.sensor}: {row.lat:.5f}, {row.lon:.5f}")
    st.download_button("Download regional summary CSV", regional_summary.to_csv(index_label="sensor").encode("utf-8"),
                       "regional_sensor_hotspot_summary.csv", "text/csv")


def render_app():
    st.set_page_config(page_title="Regional PM2.5 hotspot explorer", layout="wide")
    st.title("Regional PM2.5 hotspot explorer")
    st.caption("Local contrasts, spatial and temporal slopes, wind and persistence")
    st.info("Exploratory screening only. Candidate flags use local concentration excess and robust score. Slopes and wind are supporting descriptions, not causal proof.")
    with st.sidebar:
        st.header("Files")
        sensor = st.text_input("Sensor Parquet path", Config.sensor_path)
        weather = st.text_input("KNMI Parquet path (optional)", Config.weather_path,
            help="Leave blank to analyse concentrations without wind. A nonblank path must point to a readable file.")
        output = st.text_input("Output folder", Config.output_dir)
        save = st.checkbox("Save CSV results to output folder", value=True)
        st.header("Time window")
        start = st.text_input("Start timestamp (optional)", help="Blank selects the earliest sensor hour. Naive timestamps use the sensor timezone.")
        end = st.text_input("End timestamp, exclusive (optional)")
        days = st.number_input("Days if end is blank", min_value=1, max_value=365, value=14)
        sensor_tz = st.text_input("Timezone for naive sensor times", "UTC")
        weather_tz = st.text_input("Timezone for naive KNMI times", "UTC")
        interval_end = st.checkbox("KNMI timestamps label interval end", value=True,
            help="Shifts weather timestamps back one hour; verify your export convention.")
        with st.expander("Timestamp and performance settings"):
            sensor_unit = st.selectbox("Sensor epoch unit", ["", "s", "ms"])
            weather_unit = st.selectbox("Weather epoch unit", ["", "s", "ms"])
            batch_size = st.number_input("Parquet batch size", min_value=1_000, max_value=1_000_000, value=100_000, step=10_000)
            st.caption("Epoch units are only for Unix-epoch timestamps. Leave blank otherwise.")
        st.header("Screening neighbourhood")
        radius = st.number_input("Neighbour radius (km)", min_value=0.1, max_value=200.0, value=5.0, step=0.5)
        max_n = st.number_input("Maximum preselected neighbours", min_value=1, max_value=500, value=30)
        min_n = st.number_input("Minimum reporting neighbours", min_value=1, max_value=100, value=3)
        min_read = st.number_input("Minimum readings per sensor-hour", min_value=1, max_value=10_000, value=1)
        excess = st.number_input("Minimum excess (reported PM units)", min_value=0.0, value=5.0, step=1.0)
        robust = st.number_input("Minimum robust score", min_value=0.0, value=2.5, step=0.1)
        floor = st.number_input("Robust scale floor (reported PM units)", min_value=0.001, value=2.0, step=0.5)
        with st.expander("Slopes, wind and persistence"):
            temporal = st.number_input("Consecutive hours for temporal slope", min_value=2, max_value=168, value=3)
            weather_distance = st.number_input("Maximum weather distance (km)", min_value=0.0, value=25.0)
            resultant = st.number_input("Minimum wind-direction resultant", min_value=0.0, max_value=1.0, value=0.5, step=0.05)
            angle = st.number_input("Upwind cone half-angle (degrees)", min_value=1.0, max_value=89.0, value=45.0)
            min_upwind = st.number_input("Minimum upwind neighbours", min_value=1, max_value=100, value=2)
            sustained = st.number_input("Consecutive hours for sustained episode", min_value=1, max_value=168, value=3)
            fraction = st.number_input("Recurring: minimum flagged fraction", min_value=0.0, max_value=1.0, value=0.20, step=0.05)
            coverage = st.number_input("Recurring: minimum window coverage", min_value=0.0, max_value=1.0, value=0.50, step=0.05)
            persistence_hours = st.number_input("Recurring: minimum assessable hours", min_value=1, max_value=10_000, value=24)
            persistence_days = st.number_input("Recurring: minimum flagged UTC dates", min_value=1, max_value=365, value=3)
        run = st.button("Run analysis", type="primary", use_container_width=True)
    c = Config(sensor_path=sensor, weather_path=weather, output_dir=output, batch_size=int(batch_size),
        start=start, end=end, initial_days=int(days), sensor_tz=sensor_tz, weather_tz=weather_tz,
        sensor_epoch_unit=sensor_unit, weather_epoch_unit=weather_unit, weather_interval_end=interval_end,
        radius_km=float(radius), max_neighbors=int(max_n), min_neighbors=int(min_n), min_readings=int(min_read),
        min_excess=float(excess), robust_z=float(robust), scale_floor=float(floor), temporal_hours=int(temporal),
        max_weather_km=float(weather_distance), resultant_min=float(resultant), upwind_angle=float(angle), min_upwind=int(min_upwind),
        sustained_hours=int(sustained), persistent_fraction=float(fraction), persistence_coverage=float(coverage),
        persistence_hours=int(persistence_hours), persistence_days=int(persistence_days), save_outputs=save)
    if run:
        st.session_state.pop("analysis", None)
        for key in ["pm25_hour", "pm25_center", "pm25_center_sensor", "pm25_national"]:
            st.session_state.pop(key, None)
        log = io.StringIO()
        bar = st.progress(0)
        st.caption("Reading and aggregating first; progress updates during hourly scoring. Large files may take several minutes.")
        try:
            with st.spinner("Running analysis..."), contextlib.redirect_stdout(log):
                a = run_pipeline(c, progress=bar.progress)
            st.session_state["analysis"] = a
            st.session_state["run_log"] = log.getvalue()
            st.success("Analysis complete. Region/time controls reuse these results; they do not rerun the pipeline.")
        except Exception as exc:
            st.session_state["run_log"] = log.getvalue()
            st.error(f"Analysis failed: {exc}")
            if "Repetition level histogram" in str(exc):
                st.warning("PyArrow could not read the Parquet encoding. The UI/port is not the cause; this file may need re-exporting or a compatible PyArrow version.")
            with st.expander("Error details", expanded=True):
                st.exception(exc)
                st.code(log.getvalue())
    if "analysis" not in st.session_state:
        st.write("Check the paths in the sidebar, then select **Run analysis**. The regional maps appear once sensor data has been loaded and scored.")
        st.caption("Only the selected time window is retained in memory. Weather is optional and does not determine the concentration-analysis window.")
        if st.session_state.get("run_log"):
            with st.expander("Last run log"):
                st.code(st.session_state["run_log"])
        return
    a = st.session_state["analysis"]
    r, summary, events, wind = a["features"], a["summary"], a["events"], a["wind"]
    st.caption(f"Last completed run: {a['start']} to {a['end']} (UTC; end exclusive).")
    if asdict(c) != a["config"]:
        st.warning("Sidebar settings differ from the displayed result. Select Run analysis to apply those changes.")
    overview, episodes, sensors, wind_tab, diagnostics = st.tabs(["Regional map", "Episodes", "Sensor explorer", "Wind direction", "Method & diagnostics"])
    with overview:
        regional_overview(a)
    with episodes:
        st.caption("Episodes are consecutive flagged hours; missing or unflagged hours break a run.")
        if events.empty:
            st.info("No flagged episodes under the current screening settings.")
        else:
            st.dataframe(events, use_container_width=True)
        st.download_button("Download episode CSV", events.to_csv(index=False).encode("utf-8"), "hotspot_events.csv", "text/csv")
    with sensors:
        selected = st.selectbox("Sensor to inspect", summary.index.tolist(), key="pm25_inspect_sensor")
        fig = plot_sensor(r, selected)
        st.pyplot(fig)
        plt.close(fig)
        subset = r.loc[r.sensor.eq(selected)].sort_values("hour")
        st.dataframe(subset, use_container_width=True)
        st.download_button("Download selected sensor hours", subset.to_csv(index=False).encode("utf-8"), "selected_sensor_hours.csv", "text/csv")
        st.caption("Temporal slopes need consecutive hours. Spatial slopes need at least four reporting neighbours with adequate geometry.")
    with wind_tab:
        st.caption("Sectors indicate where wind comes from. Comparisons are descriptive associations, not causal effects.")
        st.dataframe(wind, use_container_width=True)
        st.dataframe(r.wind_status.value_counts().rename_axis("wind_status").reset_index(name="sensor_hours"), use_container_width=True)
        st.download_button("Download wind-sector CSV", wind.to_csv(index_label="wind_from_sector").encode("utf-8"), "wind_sector_summary.csv", "text/csv")
    with diagnostics:
        st.markdown("**Flags:** compare hourly sensor PM with the median of nearby reporting sensors, excluding the target. Both minimum local excess and robust score must be met. The robust scale is the larger of 1.4826 times neighbour MAD and the configured scale floor.")
        st.markdown("**Slopes:** the spatial plane is fitted to neighbours only, with inverse-square distance weights using a 0.25 km offset. Temporal slopes are least-squares trends over consecutive hours. Slopes and wind are not flag criteria.")
        st.markdown("**Regional surface:** inverse-distance weighted interpolation uses up to 12 distinct sites, a 0.05 km smoothing offset, at least three sites within the support radius, and a convex-hull mask. It is a visual estimate; it does not create new measured hotspots.")
        st.markdown("**Time and wind:** the analysis window is based on sensor availability, not weather overlap. DD=0 is calm; DD=990 is variable; missing is distinct. Interval-end shifting is configurable and must match the export.")
        st.markdown("**Limitations:** confirm reported PM units, sensor calibration, stationary locations, timezone and hourly coverage. Humidity, temperature and wind speed are not used here. Thresholds are exploratory. Recurrence describes only this run's window.")
        st.json(a["config"])
        st.code(st.session_state.get("run_log", ""))
        st.download_button("Download full sensor summary", summary.to_csv(index_label="sensor").encode("utf-8"), "sensor_hotspot_summary.csv", "text/csv")
        if a["config"]["save_outputs"]:
            st.caption(f"CSV outputs saved to: {a['config']['output_dir']}")


if __name__ == "__main__":
    render_app()
'''

# ---- Validate, write, choose a port, launch, and wait for the server ----
required = {
    "streamlit": "streamlit", "pandas": "pandas>=2", "numpy": "numpy",
    "pyarrow": "pyarrow", "sklearn": "scikit-learn", "scipy": "scipy",
    "matplotlib": "matplotlib", "folium": "folium", "streamlit_folium": "streamlit-folium",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(
        "Missing packages in THIS notebook's Python environment: " + ", ".join(missing) +
        "\nInstall them using this interpreter: " + sys.executable + " -m pip install " + " ".join(missing)
    )

# The repository is the folder (this one or a parent) that contains data/data_sensors.parquet.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "data_sensors.parquet").is_file()), None)
if ROOT is None:
    raise RuntimeError("Could not find data/data_sensors.parquet above " + str(Path.cwd()) +
                       ". Open the notebook from inside the repository and put the Zenodo file in data/.")
APP = ROOT / "output" / "pm25_hotspot_app.py"  # generated files go to <repo>/output/ (ignored by git)
APP.parent.mkdir(parents=True, exist_ok=True)
ast.parse(APP_SOURCE, filename=str(APP))

# Stop only processes tracked by previous launcher cells in this notebook,
# and only if their command line identifies this PM2.5 app.
stopped = set()
for variable in ("streamlit_process", "_pm25_streamlit_process"):
    previous = globals().get(variable)
    if isinstance(previous, subprocess.Popen) and previous.poll() is None and previous.pid not in stopped:
        args = previous.args if isinstance(previous.args, (list, tuple)) else [str(previous.args)]
        if any("pm25_hotspot_app.py" in str(arg) for arg in args):
            print(f"Stopping previous notebook-launched PM2.5 server (PID {previous.pid})...")
            previous.terminate()
            try:
                previous.wait(timeout=10)
            except subprocess.TimeoutExpired:
                previous.kill()
                previous.wait(timeout=5)
            stopped.add(previous.pid)

if APP.exists():
    backup = APP.with_name(APP.stem + "_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".py")
    backup.write_bytes(APP.read_bytes())
    print(f"Previous file backed up to: {backup}")
APP.write_text(APP_SOURCE, encoding="utf-8")
print(f"Complete app written to: {APP}")

# Prefer 8501, but let Windows choose an available port if it is occupied.
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    try:
        probe.bind(("127.0.0.1", 8501))
        port = 8501
    except OSError:
        probe.bind(("127.0.0.1", 0))
        port = probe.getsockname()[1]

LOG = APP.parent / f"pm25_streamlit_{port}.log"
url = f"http://127.0.0.1:{port}"
command = [
    sys.executable, "-m", "streamlit", "run", str(APP),
    "--server.address=127.0.0.1", f"--server.port={port}",
    "--server.headless=true", "--server.fileWatcherType=none",
    "--browser.gatherUsageStats=false",
]
environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"
with LOG.open("w", encoding="utf-8") as log_file:
    streamlit_process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT,
                                         cwd=str(APP.parent), env=environment)
_pm25_streamlit_process = streamlit_process

print(f"Starting Streamlit with notebook Python: {sys.executable}")
print(f"Chosen port: {port}; server log: {LOG}")
# Bypass HTTP proxies for the local readiness check.
opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
ready = False
for _ in range(120):
    if streamlit_process.poll() is not None:
        break
    try:
        with opener.open(url + "/_stcore/health", timeout=1) as response:
            ready = response.status == 200
        if ready:
            break
    except Exception:
        pass
    time.sleep(0.5)

if not ready:
    tail = LOG.read_text(encoding="utf-8", errors="replace")[-8000:]
    print("\nStreamlit startup log:\n" + tail)
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
    raise RuntimeError(f"Streamlit did not become ready. Review the printed log and {LOG}.")

print(f"\nStreamlit server is ready: {url}")
try:
    opened = webbrowser.open(url, new=2)
    if not opened:
        print("Browser did not open automatically; open the URL above manually.")
except Exception as exc:
    print(f"Browser could not be opened automatically ({exc}); open the URL above manually.")
try:
    from IPython.display import display, HTML
    display(HTML(f'<p><a href="{url}" target="_blank">Open the PM2.5 hotspot explorer</a></p>'))
except ImportError:
    pass
print("In the app, check the file paths and select Run analysis.")
print("If the page shows an application error, inspect the error on the page and the server log.")
print("To stop this server later: streamlit_process.terminate()")

Previous file backed up to: c:\Users\shaya\Downloads\Hackathon-Challenge-2-\output\pm25_hotspot_app_backup_20261117_193742_073417.py
Complete app written to: c:\Users\shaya\Downloads\Hackathon-Challenge-2-\output\pm25_hotspot_app.py
Starting Streamlit with notebook Python: c:\Users\shaya\Downloads\Hackathon-Challenge-2-\.venv\Scripts\python.exe
Chosen port: 8501; server log: c:\Users\shaya\Downloads\Hackathon-Challenge-2-\output\pm25_streamlit_8501.log

Streamlit server is ready: http://127.0.0.1:8501


In the app, check the file paths and select Run analysis.
If the page shows an application error, inspect the error on the page and the server log.
To stop this server later: streamlit_process.terminate()


In [5]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "streamlit",
    "pandas>=2",
    "numpy",
    "pyarrow",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "folium",
    "streamlit-folium",
])

print("Dependencies installed. Rerun the complete app-launcher cell.")

Dependencies installed. Rerun the complete app-launcher cell.
